In [1]:
%load_ext dotenv
%dotenv

In [1]:
import os
from pydantic import BaseModel, Field
from typing import Optional, List, Literal
import json
from graphrag.llm import get_openai_client
from graphrag.graph_utils import get_neo4j_driver

client = get_openai_client()
neo4j_driver = get_neo4j_driver()

In [2]:
class Location(BaseModel):
    """
    Represents a physical location including address, city, state, and country.
    """
    address: Optional[str] = Field(..., description="The street address of the location.")
    city: Optional[str] = Field(..., description="The city of the location.")
    state: Optional[str] = Field(..., description="The state or region of the location.")
    country: str = Field(..., description="The country of the location. Use the two-letter ISO standard.")


class Organization(BaseModel):
    """
    Represents an organization, including its name and location.
    """
    name: str = Field(..., description="The name of the organization.")
    location: Location = Field(..., description="The primary location of the organization.")
    role: str = Field(..., description="The role of the organization in the contract, such as 'provider', 'client', 'supplier', etc.")


class Contract(BaseModel):
    """
    Represents the key details of the contract.
    """
    contract_type: Literal["Service Agreement",
                           "Licensing Agreement",
                           "Non-Disclosure Agreement (NDA)",
                           "Partnership Agreement",
                           "Lease Agreement"] = Field(..., description="The type of contract being entered into.")
    parties: List[Organization] = Field(..., description="List of parties involved in the contract, with details of each party's role.")
    effective_date: str = Field(..., description="The date when the contract becomes effective. Use yyyy-MM-dd format.")
    term: str = Field(..., description="The duration of the agreement, including provisions for renewal or termination.")
    contract_scope: str = Field(..., description="Description of the scope of the contract, including rights, duties, and any limitations.")
    end_date: Optional[str] = Field(..., description="The date when the contract becomes expires. Use yyyy-MM-dd format.")
    total_amount: Optional[float] = Field(..., description="Total value of the contract.")
    governing_law: Optional[Location] = Field(..., description="The jurisdiction's laws governing the contract.")

In [3]:
system_message = """
You are an expert in extracting structured information from legal documents and contracts.
Identify key details such as parties involved, dates, terms, obligations, and legal definitions.
Present the extracted information in a clear, structured format. Be concise, focusing on essential
legal content and ignoring unnecessary boilerplate language."""

In [4]:
def extract(document, model="qwen2.5:14b", temperature=0):
    response = client.chat.completions.parse(model=model,
                                             temperature=temperature,
    reasoning_effort="low",
    max_completion_tokens=4096,
                                             messages=[{"role": "system", "content": system_message},
                                                       {"role": "user", "content": document}],
                                             response_format=Contract)
    message = response.choices[0].message
    if message.refusal:
        raise ValueError(f"Model refused extraction: {message.refusal}")
    return message.parsed.model_dump()    

In [5]:
# Read the file
with open('../data/license_agreement.txt', 'r') as file:
    contents = file.read()

In [6]:
data = extract(contents, model="gpt-oss:20b")
data

{'contract_type': 'Licensing Agreement',
 'parties': [{'name': 'Mortgage Logic.com, Inc.',
   'location': {'address': 'Two Venture Plaza, 2 Venture, Irvine, California 92618',
    'city': 'Irvine',
    'state': 'CA',
    'country': 'USA'},
   'role': 'Client'},
  {'name': 'TrueLink, Inc.',
   'location': {'address': '3026 South Higuera, San Luis Obispo, California 93401',
    'city': 'San Luis Obispo',
    'state': 'CA',
    'country': 'USA'},
   'role': 'Licensor'}],
 'effective_date': '1999-02-26',
 'term': '1 year (renewable annually unless terminated per Section\u202f12)',
 'contract_scope': 'TrueLink grants Client a non‑exclusive license to use the Interface for origination, underwriting, processing and funding of consumer finance receivables.  Client may also use other hardware/software at its own expense.  The license is limited to the Interface and does not grant rights to any other TrueLink IP.',
 'end_date': '2020-02-25 (subject to renewal)',
 'total_amount': None,
 'governin

In [7]:
neo4j_driver.execute_query("CREATE CONSTRAINT IF NOT EXISTS FOR (c:Contract) REQUIRE c.id IS UNIQUE;")
neo4j_driver.execute_query("CREATE CONSTRAINT IF NOT EXISTS FOR (o:Organization) REQUIRE o.name IS UNIQUE;")
neo4j_driver.execute_query("CREATE CONSTRAINT IF NOT EXISTS FOR (l:Location) REQUIRE l.fullAddress IS UNIQUE;")

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x786a83ed0050>, keys=[])

In [8]:
import_query = """WITH $data AS contract_data
// Create Contract node
MERGE (contract:Contract {id: randomUUID()})
SET contract += {
  contract_type: contract_data.contract_type,
  effective_date: contract_data.effective_date,
  term: contract_data.term,
  contract_scope: contract_data.contract_scope,
  end_date: contract_data.end_date,
  total_amount: contract_data.total_amount,
  governing_law: contract_data.governing_law.state + ' ' +
                 contract_data.governing_law.country
}
WITH contract, contract_data
// Create Party nodes and their locations
UNWIND contract_data.parties AS party
MERGE (p:Organization {name: party.name})
MERGE (loc:Location {
  fullAddress: party.location.address + ' ' +
                party.location.city + ' ' +
                party.location.state + ' ' +
                party.location.country})
SET loc += {
  address: party.location.address,
  city: party.location.city,
  state: party.location.state,
  country: party.location.country
}
// Link party to their location
MERGE (p)-[:LOCATED_AT]->(loc)
// Link parties to the contract
MERGE (p)-[r:HAS_PARTY]->(contract)
SET r.role = party.role
"""
neo4j_driver.execute_query(import_query, data=data)

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x786a83eccb00>, keys=[])